# Graph-of-Thought — Colab runner

Our GPU mode of operation (see `AGENTS.md` §31):
1. Push the repo to GitHub.
2. Run this notebook on Colab (**Runtime → Change runtime type → T4 GPU**).
3. Datasets/models come from the **Hugging Face Hub** only — there is no local data access on the remote.
4. The run writes `output/<run_id>/`; the last cell **zips and downloads** it so you can copy it back into this repo's git-ignored `output/`.

In [ ]:
# 1) Clone the repo
# Public repo:
!git clone https://github.com/arrafmousa/graph-of-thought.git

# Private repo instead? Store a GitHub token as a Colab secret named GH_TOKEN, then:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone https://{tok}@github.com/arrafmousa/graph-of-thought.git

%cd graph-of-thought

In [ ]:
# 2) (When training) install deps and authenticate Hugging Face.
# The current demo workload is pure stdlib and needs nothing installed.
# !pip install -q -r requirements.txt
# from google.colab import userdata; import os
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # add HF_TOKEN as a Colab secret

In [ ]:
# 3) Run the pipeline (live ASCII tables + sparkline graph in the cell output)
!python scripts/run.py --config configs/example_run.json

In [ ]:
# 4) Package the latest run and download it (copy the zip into this repo's output/ locally)
import glob, os
from google.colab import files
latest = sorted(glob.glob('output/*/'))[-1].rstrip('/')
run_id = os.path.basename(latest)
!zip -qr {run_id}.zip {latest}
print('Downloading', run_id + '.zip')
files.download(f'{run_id}.zip')

In [ ]:
# 5) Show the HTML dashboard inline
from IPython.display import HTML
HTML(open(os.path.join(latest, 'dashboard.html')).read())

### Alternative: import and run programmatically (no CLI)

In [ ]:
import sys; sys.path.insert(0, 'src')
from pathlib import Path
from main.run_pipeline import RunOrchestrator

orch = RunOrchestrator(
    repo_root=Path.cwd(),
    schema_path=Path('configs/schema/run_config.schema.json'),
    tracked_packages=[],
)
run_dir = orch.run(
    config_path=Path('configs/example_run.json'),
    entrypoint='colab', command='colab run',
)
print('Run:', run_dir)